
# EPOS: Global Cost vs Local Cost Simulation

### The Goal
We want to minimize the **Global Cost (GC)**, which is defined as the variance of the aggregate signal (sum of all agents' plans).
At the same time, we track the **Local Cost (LC)**, which represents the discomfort or cost to each agent for their selected plan.

### Inputs
* **Agents:** Number of entities.
* **Plans:** Number of options per agent.
* **Dimension:** The length of the plan vector (e.g., 24 hours).


In [1]:
import numpy as np
import pandas as pd
import itertools
import io
import time
import os

# ==========================================
# 1. INPUT DATA (Load from CSV)
# ==========================================
# Read CSV from File - Env var or default
csv_file = os.environ.get('EPOS_INPUT_FILE', 'input_9_4.csv')
print(f"Processing Input File: {csv_file}")

df_input = pd.read_csv(csv_file, sep=';')

# Extract Configuration from Data
NUM_AGENTS = len(df_input)
# Detect number of plans based on columns starting with 'plan_'
plan_cols = [c for c in df_input.columns if c.startswith('plan_')]
NUM_PLANS = len(plan_cols)
agent_names = [f"Agent_{id}" for id in df_input['voter_id']]
MAX_ITERATIONS = 5

np.random.seed(42)

# --- Parse Vectors and Generate Costs ---
agents = {}
for index, row in df_input.iterrows():
    name = agent_names[index]
    agents[name] = []
    
    # Process all plans
    for p in range(NUM_PLANS):
        col_name = f"plan_{p+1}"
        
        cell_value = str(row[col_name])
        
        # Check if cell contains Cost:Vector format
        if ':' in cell_value:
            parts = cell_value.split(':', 1)
            cost = float(parts[0])
            vec_str = parts[1]
        else:
            # Fallback to random cost if not provided
            cost = round(np.random.rand(), 2)
            vec_str = cell_value
        
        # Convert string "1.2, 3.4, ..." to numpy array
        try:
            vec = np.fromstring(vec_str, sep=',')
        except ValueError:
            # Hande potentially malformed strings
             vec = np.fromstring(vec_str.replace('[','').replace(']',''), sep=',')

        agents[name].append({
            'vec': vec,
            'cost': cost
        })

# Detect Dimension from first vector
PLAN_DIM = len(agents[agent_names[0]][0]['vec'])

print(f"--- DATA PARSED ---")
print(f"Agents: {NUM_AGENTS}")
print(f"Plans per Agent: {NUM_PLANS}")
print(f"Vector Dimension: {PLAN_DIM}")
print(f"Example Cost (Agent 1, Plan 1): {agents[agent_names[0]][0]['cost']}")


Processing Input File: input_uploaded_16_2026-05-14_01-24-45.csv


--- DATA PARSED ---
Agents: 4
Plans per Agent: 16
Vector Dimension: 100
Example Cost (Agent 1, Plan 1): 1.0


## Part 1: Brute Force Calculation (Ground Truth)

We calculate the GC and LC for every possible combination ($Plans^{Agents}$).
This gives us the "Map" of the entire solution space.

In [2]:
print("\n" + "="*60)
print(f"PART 1: BRUTE FORCE ANALYSIS ({NUM_PLANS}^{NUM_AGENTS} = {NUM_PLANS**NUM_AGENTS} Solutions)")
print("="*60)
if NUM_PLANS**NUM_AGENTS > 10000000:
    print("WARNING: Large number of combinations. This may take a long time.")

start_time_bf = time.time()  # Start Timer

combinations = list(itertools.product(range(NUM_PLANS), repeat=NUM_AGENTS))
bf_data = []

for combo in combinations:
    vecs = [agents[agent_names[i]][combo[i]]['vec'] for i in range(NUM_AGENTS)]
    costs = [agents[agent_names[i]][combo[i]]['cost'] for i in range(NUM_AGENTS)]
    
    g_sum = np.sum(vecs, axis=0)
    gc = np.var(g_sum)
    avg_lc = sum(costs) / NUM_AGENTS
    
    bf_data.append({
        "Indices": combo,
        "GC": gc,
        "LC": avg_lc
    })

bf_duration = time.time() - start_time_bf  # End Timer
print(f"Brute Force Calculation took: {bf_duration:.4f} seconds")

df_bf = pd.DataFrame(bf_data)
# Sort to find global optimum
df_sorted = df_bf.sort_values(by="GC")
global_optimum_row = df_sorted.iloc[0]

GLOBAL_MIN_GC = global_optimum_row['GC']
GLOBAL_BEST_COMBO = global_optimum_row['Indices']

print(f"Global Optimum Found: GC={GLOBAL_MIN_GC:.6f}")
print(f"Best Configuration: {GLOBAL_BEST_COMBO}")
display(df_bf)


PART 1: BRUTE FORCE ANALYSIS (16^4 = 65536 Solutions)


Brute Force Calculation took: 20.4544 seconds


Global Optimum Found: GC=2.037223
Best Configuration: (6, 7, 13, 11)


,Indices,GC,LC
0,"(0, 0, 0, 0)",3.621281,1.00
1,"(0, 0, 0, 1)",3.682921,1.25
2,"(0, 0, 0, 2)",3.317768,1.50
3,"(0, 0, 0, 3)",4.105359,1.75
4,"(0, 0, 0, 4)",3.596170,2.00
...,...,...,...
65531,"(15, 15, 15, 11)",4.783961,15.00
65532,"(15, 15, 15, 12)",4.175723,15.25
65533,"(15, 15, 15, 13)",4.918440,15.50
65534,"(15, 15, 15, 14)",4.703039,15.75



## Part 2: Iterative Simulation (The Algorithm)

Now we simulate the agents running EPOS.
* **Initialization:** Everyone picks Plan 0.
* **Step:** Each agent, one by one, looks at the current global sum and checks if switching plans reduces the variance.
* **Rule:** They switch ONLY if Variance decreases.


In [3]:
print()
print("="*60)
print(f"PART 2: RUNNING EPOS FROM ALL {len(combinations)} STARTING POSITIONS")
print("="*60)

start_time_epos = time.time()  # Start Timer

simulation_results = []
iteration_history_rows = []

def get_stats(plan_indices):
    vecs = [agents[agent_names[i]][plan_indices[i]]['vec'] for i in range(NUM_AGENTS)]
    costs = [agents[agent_names[i]][plan_indices[i]]['cost'] for i in range(NUM_AGENTS)]
    g_sum = np.sum(vecs, axis=0)
    return np.var(g_sum), sum(costs)/NUM_AGENTS, g_sum

# Iterate through every possible starting state
for start_combo in combinations:

    # Init plans
    current_plans = {name: start_combo[i] for i, name in enumerate(agent_names)}

    # Init Global Sum
    _, _, current_global_sum = get_stats(list(current_plans.values()))

    # Run EPOS Logic
    for it in range(MAX_ITERATIONS):
        # Track per-agent local costs at the start of each iteration
        iter_row = {"Start_State": str(start_combo), "Iteration": it + 1}
        for i, name in enumerate(agent_names):
            iter_row[f"Agent {i}"] = agents[name][current_plans[name]]['cost']
        iteration_history_rows.append(iter_row)

        changes = 0

        # Agents take turns
        for name in agent_names:
            curr_idx = current_plans[name]
            curr_vec = agents[name][curr_idx]['vec']

            # Partial Sum (Subtract my current contribution)
            partial_sum = current_global_sum - curr_vec

            best_idx = curr_idx
            best_var = np.var(current_global_sum)

            # Check all plans
            for p_idx in range(NUM_PLANS):
                if p_idx == curr_idx: continue

                cand_vec = agents[name][p_idx]['vec']
                pot_sum = partial_sum + cand_vec
                pot_var = np.var(pot_sum)

                # Greedy Selection (Variance only)
                if pot_var < best_var:
                    best_var = pot_var
                    best_idx = p_idx

            # Update if improved
            if best_idx != curr_idx:
                current_plans[name] = best_idx
                current_global_sum = partial_sum + agents[name][best_idx]['vec']
                changes += 1

        if changes == 0:
            break

    # Record Result
    final_indices = tuple(current_plans.values())
    final_gc, final_lc, _ = get_stats(final_indices)

    # Check if we hit the global optimum (using isclose for float comparison)
    is_optimum = np.isclose(final_gc, GLOBAL_MIN_GC)

    simulation_results.append({
        "Start_Indices": str(start_combo),
        "Final_Indices": str(final_indices),
        "Final_GC": round(final_gc, 6),
        "Final_LC": round(final_lc, 4),
        "Hit_Global_Optimum": is_optimum
    })

epos_duration = time.time() - start_time_epos  # End Timer
print(f"EPOS Simulation took: {epos_duration:.4f} seconds")

# ==========================================
# 5. RESULTS SUMMARY
# ==========================================
df_sim = pd.DataFrame(simulation_results)
df_iter = pd.DataFrame(iteration_history_rows)

print("--- SIMULATION SUMMARY ---")
total_runs = len(df_sim)
success_runs = df_sim['Hit_Global_Optimum'].sum()
success_rate = (success_runs / total_runs) * 100

print(f"Total Starting States: {total_runs}")
print(f"Global Optimum GC: {GLOBAL_MIN_GC:.6f}")
print(f"Success Rate: {success_rate:.2f}% ({success_runs}/{total_runs} reached optimum)")


PART 2: RUNNING EPOS FROM ALL 65536 STARTING POSITIONS


EPOS Simulation took: 653.0461 seconds


--- SIMULATION SUMMARY ---
Total Starting States: 65536
Global Optimum GC: 2.037223
Success Rate: 9.13% (5984/65536 reached optimum)


In [4]:
import os
import shutil

# Determine output directory name
# Expecting EPOS_OUTPUT_FOLDER from environment, else default to old scheme
output_dir = os.environ.get('EPOS_OUTPUT_FOLDER')
if not output_dir:
    output_dir = f"{NUM_AGENTS}_agent_{NUM_PLANS}_plan"

os.makedirs(output_dir, exist_ok=True)

# Save simulation results
df_sim.to_csv(os.path.join(output_dir, "epossimulationresults.csv"), index=False)
print(f"Saved simulation results to: {output_dir}/epossimulationresults.csv")

# Save Solution Wise Results
df_bf.to_csv(os.path.join(output_dir, "solutionwiseresults.csv"), index=False)
print(f"Saved solution wise results to: {output_dir}/solutionwiseresults.csv")

# Save Iteration Cost History
df_iter.to_csv(os.path.join(output_dir, "iteration_cost_history.csv"), index=False)
print(f"Saved iteration cost history to: {output_dir}/iteration_cost_history.csv")

# Move the Input CSV file to the output directory for archiving/cleanup
try:
    if os.path.exists(csv_file):
        shutil.move(csv_file, output_dir)
        print(f"Moved input file to: {output_dir}/{os.path.basename(csv_file)}")
except Exception as e:
    print(f"Warning: Could not move input file: {e}")

# Log the summary stats to a file (using .md for formatting)
log_file = os.path.join(output_dir, "simulation_log.md")
with open(log_file, "w") as f:
    f.write("# EPOS SIMULATION LOG\n\n")
    f.write(f"- Date: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"- Input Source: {csv_file}\n")
    f.write(f"- Brute Force Duration: {bf_duration:.4f} seconds\n")
    f.write(f"- EPOS Simulation Duration: {epos_duration:.4f} seconds\n\n")
    
    # Brute Force Analysis Section
    f.write("## PART 1: BRUTE FORCE ANALYSIS\n")
    f.write(f"- Total Combinations Analyzed: {NUM_PLANS}^{NUM_AGENTS} = {NUM_PLANS**NUM_AGENTS}\n")
    f.write(f"- Global Optimum GC: {GLOBAL_MIN_GC:.6f}\n")
    f.write(f"- Best Configuration (Indices): {GLOBAL_BEST_COMBO}\n")
    f.write(f"- Corresponding LC: {global_optimum_row['LC']:.4f}\n\n")
    
    # Statistics about GC distribution
    f.write("### GC Distribution Statistics\n")
    f.write(f"- Min GC: {df_bf['GC'].min():.6f}\n")
    f.write(f"- Max GC: {df_bf['GC'].max():.6f}\n")
    f.write(f"- Mean GC: {df_bf['GC'].mean():.6f}\n")
    f.write(f"- Median GC: {df_bf['GC'].median():.6f}\n")
    f.write(f"- Std Dev GC: {df_bf['GC'].std():.6f}\n\n")
    
    # EPOS Simulation Section
    f.write("## PART 2: EPOS SIMULATION RESULTS\n")
    f.write(f"- Total Starting States Tested: {len(df_sim)}\n")
    f.write(f"- Max Iterations per Run: {MAX_ITERATIONS}\n")
    success_count = df_sim['Hit_Global_Optimum'].sum()
    rate = (success_count / len(df_sim)) * 100
    f.write(f"- Success Rate: {rate:.2f}% ({success_count}/{len(df_sim)} reached optimum)\n\n")
    
    # Convergence Statistics
    f.write("### Convergence Statistics\n")
    f.write(f"- Unique Final States: {df_sim['Final_Indices'].nunique()}\n")
    f.write(f"- Final GC Range: [{df_sim['Final_GC'].min():.6f}, {df_sim['Final_GC'].max():.6f}]\n")
    f.write(f"- Final LC Range: [{df_sim['Final_LC'].min():.4f}, {df_sim['Final_LC'].max():.4f}]\n")
    
print(f"Saved comprehensive log to: {log_file}")

Saved simulation results to: 4_agent_16_plan_uploaded_2026-05-14_01-24-45/epossimulationresults.csv


Saved solution wise results to: 4_agent_16_plan_uploaded_2026-05-14_01-24-45/solutionwiseresults.csv


Saved iteration cost history to: 4_agent_16_plan_uploaded_2026-05-14_01-24-45/iteration_cost_history.csv
Moved input file to: 4_agent_16_plan_uploaded_2026-05-14_01-24-45/input_uploaded_16_2026-05-14_01-24-45.csv
Saved comprehensive log to: 4_agent_16_plan_uploaded_2026-05-14_01-24-45\simulation_log.md
